# 🧵 Manual Multi-Tool Orchestration — How an LLM Plans & Your Code Executes

### Dinesh AI Academy | Day 3 — Tools & Workflows (Workshop companion)

**Learning objective:**
See, in plain Python, the exact mechanism that AI agent frameworks (LangChain, CrewAI, LangGraph, and the "automatic function calling" feature built into the Gemini/OpenAI SDKs) hide behind a decorator. No framework is used anywhere in this notebook — every step stays visible.

**The scenario:** one user message that needs **more than one tool**, in a **specific order**. We will:

1. Give Gemini a custom system instruction + a menu of tools.
2. Send **one** user query that genuinely needs multiple tools.
3. Inspect the **ordered list** of tool-call requests Gemini returns.
4. Use plain Python — a loop + a dictionary lookup — to map each requested tool name to a real Python function, and execute them in that order.
5. Send every result back to Gemini so it can write one combined, natural-language answer.

```text
User query (needs 3 things, in order)
        ↓
Gemini reads: query + system instruction + tool menu
        ↓
Gemini returns an ORDERED LIST of tool-call requests
   [ call_1, call_2, call_3 ]
        ↓
Plain Python:
   for call in list:
       function = TOOL_REGISTRY[call.name]   # <- the "framework magic"
       result   = function(**call.args)
        ↓
All results sent back to Gemini
        ↓
Gemini writes ONE final answer
```

> **This *is* how frameworks do it.** `AgentExecutor`, `@tool` decorators, "automatic function calling" — underneath, they all keep a `{name: callable}` dictionary and loop over the model's returned tool calls, exactly like this notebook does by hand.

## 1. Install the Gemini SDK

We use Google's official `google-genai` Python SDK — the same one used throughout this bootcamp.

In [1]:
# In Google Colab, run this cell once. Locally, run it inside your venv.
!pip -q install -U google-genai

In [5]:
import json
from pathlib import Path
from datetime import datetime, timezone


def save_llm_response(response, filepath: str = None, directory: str = "llm_logs",
                       include_http_metadata: bool = False) -> str:
    """
    Save a full LLM API response object to a JSON file — every field the SDK
    returned, not just response.text. Works with any pydantic-based SDK
    response (google-genai, openai>=1.x); falls back gracefully otherwise.

    Args:
        response: the raw response object, e.g. from
            client.models.generate_content(...).
        filepath: exact path to write to. If omitted, an auto-named,
            timestamped file is created inside `directory`.
        directory: folder used when `filepath` isn't given (created if missing).
        include_http_metadata: keep raw HTTP transport headers too
            (off by default — noise, not really part of "the response").

    Returns:
        The path the response was written to.
    """
    # ---- Turn the response object into a plain JSON-safe dict ----
    if hasattr(response, "model_dump"):
        # Pydantic v2 models (google-genai, openai>=1.x) — correctly handles
        # bytes (base64-encodes them), enums, and nested objects like
        # usage_metadata and function_call arguments.
        data = response.model_dump(mode="json", exclude_none=True)
    elif hasattr(response, "to_dict"):
        data = response.to_dict()
    elif hasattr(response, "__dict__"):
        data = vars(response)
    else:
        data = {"raw": str(response)}

    if not include_http_metadata:
        data.pop("sdk_http_response", None)  # raw HTTP headers, not model output

    # ---- Work out where to write it ----
    if filepath is None:
        Path(directory).mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
        filepath = str(Path(directory) / f"response_{timestamp}.json")
    else:
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    return filepath

## 2. Load the Gemini API key and create the client

The key lives in a `.env` file as `GAISTUDIO_API_KEY=...` (or in Colab Secrets, if you're running there).
It is **never** hard-coded in this notebook — that's the one rule every AI engineer breaks exactly once before learning it the hard way.

In [6]:
from google import genai
from google.genai import types
import os

def get_secret(key_name: str) -> str:
    """Works in both Google Colab (Secrets) and a local .env file — same helper used across this bootcamp's notebooks."""
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

client = genai.Client(api_key=GAISTUDIO_API_KEY)

# Change this if your account has access to a different Gemini model.
MODEL = "gemini-3.5-flash-lite"


def execute_llm_model_n_generate_content(contents, config):
    """Thin wrapper around generate_content that retries on a free-tier rate
    limit (HTTP 429) or a transient server hiccup (HTTP 5xx). Every call in
    this notebook goes through here so a classroom demo doesn't die on a
    quota blip or a momentary "high demand" error."""
    from google.genai import errors as genai_errors
    import time

    for attempt in range(3):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            transient = "RESOURCE_EXHAUSTED" in str(e) or "UNAVAILABLE" in str(e) or isinstance(e, genai_errors.ServerError)
            if transient and attempt < 2:
                wait_seconds = 20 * (attempt + 1)
                print(f"Temporary API error — waiting {wait_seconds}s before retrying...")
                time.sleep(wait_seconds)
            else:
                raise


print("✅ Gemini client ready. Model:", MODEL)

✅ Gemini client ready. Model: gemini-3.5-flash-lite


## 3. Step 1 — Write the real Python functions first

Before any of this is "AI", these are just ordinary functions. Nothing here talks to Gemini yet.
We'll expose four of them, deliberately unrelated to each other, so a query can require any combination of them.

In [7]:
from datetime import datetime
from zoneinfo import ZoneInfo


def get_weather_fn(city: str) -> dict:
    """Demo only — replace with a real weather API in production."""
    fake_conditions = {"Tokyo": "18°C, clear", "Paris": "14°C, light rain", "New York": "9°C, windy"}
    return {"city": city, "conditions": fake_conditions.get(city, "22°C, sunny (demo default)")}


def calculate_fn(a: float, b: float, operation: str) -> float:
    """A plain arithmetic function — this is the 'give the LLM a reliable calculator' pattern."""
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    if operation == "percentage":
        # a percent of b, e.g. calculate(18, 4500, "percentage") -> 18% of 4500
        return (a / 100) * b
    raise ValueError(f"Unsupported operation: {operation}")


def convert_currency_fn(amount: float, from_currency: str, to_currency: str) -> dict:
    """Demo only — fixed exchange rates, no live API call."""
    fake_rates_to_usd = {"USD": 1.0, "EUR": 0.92, "INR": 83.0, "GBP": 0.79}
    if from_currency not in fake_rates_to_usd or to_currency not in fake_rates_to_usd:
        return {"error": f"Unsupported currency pair: {from_currency} -> {to_currency}"}
    usd_amount = amount / fake_rates_to_usd[from_currency]
    converted = usd_amount * fake_rates_to_usd[to_currency]
    return {"amount": amount, "from": from_currency, "to": to_currency, "converted": round(converted, 2)}


def get_current_time_fn(timezone: str) -> dict:
    """Real system clock — the one tool here that isn't faked."""
    now = datetime.now(ZoneInfo(timezone))
    return {"timezone": timezone, "time": now.strftime("%Y-%m-%d %H:%M:%S")}


# Quick sanity check — call them directly, no Gemini involved yet.
print(get_weather_fn("Tokyo"))
print(calculate_fn(18, 4500, "percentage"))
print(convert_currency_fn(250, "USD", "EUR"))
print(get_current_time_fn("Asia/Tokyo"))

{'city': 'Tokyo', 'conditions': '18°C, clear'}
810.0
{'amount': 250, 'from': 'USD', 'to': 'EUR', 'converted': 230.0}
{'timezone': 'Asia/Tokyo', 'time': '2026-09-23 21:42:48'}


## 4. Step 2 — Describe each function to Gemini (tool declarations)

Gemini cannot read our Python source code. We describe each function as a **name + description + argument schema** — a menu of capabilities.
The `description` field is doing real work here: it's the only information Gemini has to decide *when* a tool is relevant.

In [8]:
weather_fn_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get the current weather conditions for a city.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={"city": types.Schema(type=types.Type.STRING, description="City name")},
        required=["city"],
    ),
)

calculate_fn_declaration = types.FunctionDeclaration(
    name="calculate",
    description="Perform a mathematical calculation: add, subtract, multiply, divide, or percentage (a% of b).",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "a": types.Schema(type=types.Type.NUMBER, description="First number"),
            "b": types.Schema(type=types.Type.NUMBER, description="Second number"),
            "operation": types.Schema(
                type=types.Type.STRING,
                description="One of: add, subtract, multiply, divide, percentage",
            ),
        },
        required=["a", "b", "operation"],
    ),
)

convert_currency_fn_declaration = types.FunctionDeclaration(
    name="convert_currency",
    description="Convert an amount of money from one currency to another.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "amount": types.Schema(type=types.Type.NUMBER, description="Amount to convert"),
            "from_currency": types.Schema(type=types.Type.STRING, description="3-letter currency code, e.g. USD"),
            "to_currency": types.Schema(type=types.Type.STRING, description="3-letter currency code, e.g. EUR"),
        },
        required=["amount", "from_currency", "to_currency"],
    ),
)

time_fn_declaration = types.FunctionDeclaration(
    name="get_current_time",
    description="Get the current date and time for an IANA timezone such as Asia/Tokyo or Europe/Paris.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={"timezone": types.Schema(type=types.Type.STRING, description="IANA timezone name")},
        required=["timezone"],
    ),
)

# Bundle every declaration into one Tool — this whole object is the "menu" Gemini sees.
tool = types.Tool(
    function_declarations=[
        weather_fn_declaration,
        calculate_fn_declaration,
        convert_currency_fn_declaration,
        time_fn_declaration,
    ]
)

print("✅ Tool object created with 4 function declarations.\r\n", tool)
print("4 tools declared:", [d.name for d in tool.function_declarations])

✅ Tool object created with 4 function declarations.
 retrieval=None computer_use=None file_search=None google_search=None google_maps=None code_execution=None enterprise_web_search=None function_declarations=[FunctionDeclaration(
  description='Get the current weather conditions for a city.',
  name='get_weather',
  parameters=Schema(
    properties={
      'city': Schema(
        description='City name',
        type=<Type.STRING: 'STRING'>
      )
    },
    required=[
      'city',
    ],
    type=<Type.OBJECT: 'OBJECT'>
  )
), FunctionDeclaration(
  description='Perform a mathematical calculation: add, subtract, multiply, divide, or percentage (a% of b).',
  name='calculate',
  parameters=Schema(
    properties={
      'a': Schema(
        description='First number',
        type=<Type.NUMBER: 'NUMBER'>
      ),
      'b': Schema(
        description='Second number',
        type=<Type.NUMBER: 'NUMBER'>
      ),
      'operation': Schema(
        description='One of: add, subtract,

## 5. Step 3 — The Python ↔ tool-name map

Gemini will hand us back a **string** (a tool name) and a dict of arguments. Something has to turn that string into the actual Python function to call.

That "something" is not magic — it's a dictionary.

> **This single dictionary is the entire "engine" behind every agent framework's tool execution.** A `{name: callable}` lookup table from the string the model returns to the real function. LangChain's `AgentExecutor`, CrewAI's tool dispatch, and the SDKs' own "automatic function calling" all keep one of these internally.

In [9]:
TOOL_REGISTRY = {
    "get_weather": get_weather_fn,
    "calculate": calculate_fn,
    "convert_currency": convert_currency_fn,
    "get_current_time": get_current_time_fn,
}

print("Tool registry:", list(TOOL_REGISTRY.keys()))

Tool registry: ['get_weather', 'calculate', 'convert_currency', 'get_current_time']


## 6. Step 4 — A custom system instruction that asks for the FULL plan up front

By default a model may request one tool, wait, then decide the next step. Here we want something more specific for this demo: **identify every tool call the whole request needs, and return them together, in the order they should run.**

This is only possible when the tools don't depend on each other's results — which is exactly the kind of request we'll ask below.

In [10]:
SYSTEM_INSTRUCTION = (
    "You are a precise assistant with access to tools. "
    "When the user's request requires more than one action, identify EVERY tool call needed "
    "to fully satisfy the request, and return them all together, in the exact order the user "
    "asked for them. Only request a tool if it is actually necessary. "
    "If a tool call genuinely depends on the result of another tool you haven't run yet, "
    "request only what you can safely determine now — you will get another turn."
)

config = types.GenerateContentConfig(
    tools=[tool],
    system_instruction=SYSTEM_INSTRUCTION,
    temperature=0,  # deterministic tool selection for a repeatable classroom demo
)

print("System instruction and tool configuration ready.")

System instruction and tool configuration ready.


## 7. Step 5 — Ask one question that needs three *independent* tools, in order

Each of these three tools can be planned without waiting on another tool's result — current time, a percentage calculation, and a weather lookup have nothing to do with each other. That's deliberate: it's what makes a single-turn, ordered "plan" possible.

In [11]:
user_prompt = (
    "First tell me the current time in Tokyo. "
    "Then calculate 18 percent of 4500. "
    "Finally, tell me the weather in Paris."
)

response = execute_llm_model_n_generate_content(contents=user_prompt, config=config)

saved_path = save_llm_response(response)
print("Saved to:", saved_path)

print("USER:", user_prompt)
print()
print("RAW RESPONSE PARTS:")
for i, part in enumerate(response.candidates[0].content.parts):
    print(f"  Part {i}: text={part.text!r}  function_call={part.function_call}")

Saved to: llm_logs\response_20260923_124308_132224.json
USER: First tell me the current time in Tokyo. Then calculate 18 percent of 4500. Finally, tell me the weather in Paris.

RAW RESPONSE PARTS:
  Part 0: text=None  function_call=id='call_283594' args={'timezone': 'Asia/Tokyo'} name='get_current_time' partial_args=None will_continue=None
  Part 1: text=None  function_call=id='call_283595' args={'a': 18, 'operation': 'percentage', 'b': 4500} name='calculate' partial_args=None will_continue=None
  Part 2: text=None  function_call=id='call_283596' args={'city': 'Paris'} name='get_weather' partial_args=None will_continue=None


## 8. Step 6 — Extract the ordered list of tool calls (plain Python, no SDK magic)

Just a list comprehension over the parts Gemini returned. The **order of this list is the order Gemini decided to run them in.**

In [12]:
# Goal: look through everything Gemini said in this turn, and pull out
# only the parts where it asked to call a tool (skip plain text parts).

requested_calls = []  # we'll collect the tool-call requests here, in order

# response.candidates[0]        -> Gemini's first (and usually only) answer option
# .content.parts                -> the list of "pieces" that make up that answer
#                                   (each piece is either plain text OR a tool-call request)
all_parts = response.candidates[0].content.parts

for part in all_parts:
    # Every part has a `.function_call` attribute.
    # It's None if this part is just plain text.
    # It holds a FunctionCall object if Gemini is requesting a tool.
    if part.function_call:
        requested_calls.append(part.function_call)

print(f"Gemini asked for {len(requested_calls)} tool call(s):")
for call in requested_calls:
    print(" -", call.name, dict(call.args))

print(f"Gemini asked for {len(requested_calls)} tool call(s), in this order:\n")
for i, call in enumerate(requested_calls, start=1):
    print(f"  {i}. {call.name}({dict(call.args)})")

Gemini asked for 3 tool call(s):
 - get_current_time {'timezone': 'Asia/Tokyo'}
 - calculate {'a': 18, 'operation': 'percentage', 'b': 4500}
 - get_weather {'city': 'Paris'}
Gemini asked for 3 tool call(s), in this order:

  1. get_current_time({'timezone': 'Asia/Tokyo'})
  2. calculate({'a': 18, 'operation': 'percentage', 'b': 4500})
  3. get_weather({'city': 'Paris'})


## 9. Step 7 — Select + execute each tool, in order (the core exercise)

For every requested call:
1. **Select** — look up `call.name` in `TOOL_REGISTRY` to get the real function.
2. **Execute** — call it with `call.args`.
3. Keep the result so we can send it all back to Gemini in one shot.

This is the exact point where you'd normally reach for a framework. Here it's a dozen lines of plain Python.

In [13]:
execution_log = []        # will store one entry per tool we ran: (name, args, result)
tool_response_parts = []  # will store the "answer" objects we send back to Gemini

step_number = 1  # just a counter for printing "Step 1", "Step 2", ...

for call in requested_calls:

    # ---------- 1. Read what Gemini is asking for ----------
    tool_name = call.name          # e.g. "get_weather"
    tool_args = dict(call.args)    # e.g. {"city": "Paris"}

    # ---------- 2. SELECT: find the real Python function with that name ----------
    # TOOL_REGISTRY is just a dictionary, e.g.:
    #   {"get_weather": get_weather, "calculate": calculate, ...}
    # .get(tool_name) looks up "get_weather" and returns the ACTUAL function object.
    # If the name isn't in the dictionary, .get() returns None instead of crashing.
    matching_function = TOOL_REGISTRY.get(tool_name)

    # ---------- 3. EXECUTE: call that function with the arguments Gemini gave us ----------
    if matching_function is None:
        # Gemini asked for a tool we don't actually have — don't crash, just report it.
        result = {"error": f"Unknown tool requested: {tool_name}"}
    else:
        # This calls the function using the arguments dict.
        # tool_args = {"city": "Paris"} becomes the same as calling:
        #   get_weather(city="Paris")
        result = matching_function(**tool_args)

    # ---------- 4. Keep a record + print progress ----------
    execution_log.append((tool_name, tool_args, result))
    print(f"Step {step_number}: {tool_name}({tool_args})")
    print(f"         -> {result}\n")
    step_number += 1

    # ---------- 5. Package the result the way Gemini expects it back ----------
    # Gemini wants a dict. Some of our functions return a plain number or string
    # (like calculate() returning 810), so we wrap those in a dict here.
    if isinstance(result, dict):
        response_payload = result
    else:
        response_payload = {"result": result}

    tool_response_part = types.Part.from_function_response(
        name=tool_name,
        response=response_payload,
    )
    tool_response_parts.append(tool_response_part)

print("All tool calls executed. Sending results back to Gemini for final answer generation...")
print("Execution log:", execution_log)
print("Tool response parts:", tool_response_parts)

Step 1: get_current_time({'timezone': 'Asia/Tokyo'})
         -> {'timezone': 'Asia/Tokyo', 'time': '2026-09-23 21:43:08'}

Step 2: calculate({'a': 18, 'operation': 'percentage', 'b': 4500})
         -> 810.0

Step 3: get_weather({'city': 'Paris'})
         -> {'city': 'Paris', 'conditions': '14°C, light rain'}

All tool calls executed. Sending results back to Gemini for final answer generation...
Execution log: [('get_current_time', {'timezone': 'Asia/Tokyo'}, {'timezone': 'Asia/Tokyo', 'time': '2026-09-23 21:43:08'}), ('calculate', {'a': 18, 'operation': 'percentage', 'b': 4500}, 810.0), ('get_weather', {'city': 'Paris'}, {'city': 'Paris', 'conditions': '14°C, light rain'})]
Tool response parts: [Part(
  function_response=FunctionResponse(
    name='get_current_time',
    response={
      'time': '2026-09-23 21:43:08',
      'timezone': 'Asia/Tokyo'
    }
  )
), Part(
  function_response=FunctionResponse(
    name='calculate',
    response={
      'result': 810.0
    }
  )
), Part(
 

## 10. Step 8 — Send every result back so Gemini writes ONE final answer

We rebuild the conversation: the original user message, Gemini's own tool-call turn (unchanged), then one more turn carrying **all** the function results together, in the same order as the calls. (The Gemini API expects function results back on the `user` role, not a separate `tool` role.)

In [14]:
# We're rebuilding the full conversation as a list of "turns", like a chat
# transcript, so Gemini has everything it needs to write ONE final answer.
# Gemini has no memory between API calls — if we skip any turn below,
# it won't know what question it's answering or what it already asked for.
conversation = [

    # Turn 1 — WHO: you (the user)
    # WHAT: your original question, exactly as first sent
    # WHY:  Gemini needs to remember what was actually asked
    types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),

    # Turn 2 — WHO: Gemini itself
    # WHAT: Gemini's own earlier reply — the one where it requested the 3 tool calls
    # WHY:  without this, the results in Turn 3 would be "random numbers" with
    #       no record of which tools Gemini asked for or in what order
    response.candidates[0].content,
    
    # Turn 3 — WHO: you again (the application, on your Python side)
    # WHAT: the REAL results your TOOL_REGISTRY functions just computed
    # WHY:  this is the actual payoff — it's what lets Gemini turn raw tool
    #       output into a natural-language answer instead of guessing
    # NOTE: the Gemini API requires function results to be sent on the
    #       "user" role — there is no separate "tool" role, even though
    #       conceptually this turn is "the tool talking back"
    types.Content(role="user", parts=tool_response_parts),
]

# Send the WHOLE transcript above in one call. Gemini reads all 3 turns,
# matches each tool result back to the tool call that requested it, and
# writes a single combined answer covering all of them.
final_response = execute_llm_model_n_generate_content(contents=conversation, config=config)

print("FINAL ANSWER:\n")
print(final_response.text)  # Gemini's one natural-language reply, e.g.
                             # "The time in Tokyo is... 18% of 4500 is 810..."

FINAL ANSWER:

The current time in Tokyo is 21:43 on September 23, 2026. 

18 percent of 4500 is 810.

The weather in Paris is 14°C with light rain.


## 11. Wrap it into one reusable function

Steps 5–10 are always the same shape: ask → (if tools requested) select + execute in order → send results back → repeat until Gemini answers in plain text.

Wrapping that shape into a function gives us a small, honest "agent loop" — the same loop every framework runs, just without the framework. The `max_turns` cap is the manual version of a dependent multi-step case (like currency conversion feeding into a percentage calculation), where Gemini can only plan one step at a time.

In [15]:
def run_multi_tool_query(user_prompt: str, max_turns: int = 4, verbose: bool = True) -> str:
    """
    This function repeats the "ask Gemini -> run any requested tools -> send
    results back" cycle automatically, instead of you doing it by hand like
    in the earlier cells. It stops as soon as Gemini gives a plain-text
    answer (no more tools needed), or after `max_turns` tries as a safety net.
    """

    # types.Content = one "turn" in the conversation (has a role + parts).
    # types.Part.from_text(...) = one piece of plain text inside that turn.
    # We wrap the plain string `user_prompt` in these SDK objects because
    # that's the exact shape the Gemini API requires — it won't accept a
    # bare string once we're building a multi-turn conversation like this.
    conversation = [
        types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)])
    ]

    step_number = 1  # just for the printed "Step 1, Step 2..." labels — not sent to Gemini

    # WHY max_turns exists: if Gemini kept requesting tools forever (a bug,
    # a bad prompt, a model quirk), this loop would run forever with nothing
    # to stop it. Capping it at `max_turns` guarantees the function always
    # returns something instead of hanging.
    for turn_number in range(1, max_turns + 1):

        # ---------- Ask Gemini, using everything said so far ----------
        # `config` is the GenerateContentConfig built earlier (tools + system
        # instruction) — same config reused on every call in this loop.
        response = execute_llm_model_n_generate_content(contents=conversation, config=config)

        saved_path = save_llm_response(response)
        print("Saved to:", saved_path)
        
        # response.candidates is a LIST of possible replies (Gemini can be
        # asked to generate several alternative answers at once). We only
        # ever asked for one, so [0] is always "the" reply here.
        model_turn = response.candidates[0].content

        # Add Gemini's reply to the transcript so it's remembered next round
        # — without this line, Gemini would "forget" it ever asked for a tool.
        conversation.append(model_turn)

        # ---------- Check whether Gemini asked for any tools ----------
        requested_calls = []
        for part in model_turn.parts:
            if part.function_call:
                requested_calls.append(part.function_call)

        if len(requested_calls) == 0:
            # Gemini didn't ask for a tool this time — that means it already
            # has everything it needs, and response.text IS the final answer.
            return response.text

        if verbose:
            # `verbose` just toggles these progress prints on/off — set it to
            # False if you want run_multi_tool_query() to work silently and
            # only hand back the final text.
            print(f"--- Turn {turn_number}: Gemini requested {len(requested_calls)} tool call(s) ---")

        # ---------- Run every requested tool, same SELECT + EXECUTE as before ----------
        turn_response_parts = []

        for call in requested_calls:
            tool_name = call.name

            # call.args comes back from the SDK as a special mapping object,
            # not a plain Python dict — wrapping it in dict(...) converts it
            # to an ordinary dict so **tool_args below works reliably.
            tool_args = dict(call.args)

            # SELECT: TOOL_REGISTRY is a plain dict of {"name": function}.
            # .get(tool_name) looks it up and returns None if it's missing —
            # unlike TOOL_REGISTRY[tool_name], .get() won't crash the whole
            # notebook if Gemini ever asks for a tool name we don't have.
            matching_function = TOOL_REGISTRY.get(tool_name)

            # EXECUTE: call it, or report that the name wasn't recognized.
            if matching_function is None:
                result = {"error": f"Unknown tool: {tool_name}"}
            else:
                # The ** in front of tool_args "unpacks" the dict into
                # keyword arguments. If tool_args = {"city": "Paris"}, this
                # line is exactly the same as writing matching_function(city="Paris").
                result = matching_function(**tool_args)

            if verbose:
                print(f"Step {step_number}: {tool_name}({tool_args}) -> {result}")
            step_number += 1

            # Gemini expects a dict back — wrap plain numbers/strings in one.
            if isinstance(result, dict):
                payload = result
            else:
                payload = {"result": result}

            # types.Part.from_function_response(...) builds the specific SDK
            # object Gemini expects for "here's the result of the tool you
            # asked for" — pairing it back up with `tool_name` so Gemini knows
            # which of its requests this result answers.
            turn_response_parts.append(
                types.Part.from_function_response(name=tool_name, response=payload)
            )

        # Add this round's results to the transcript as ANOTHER "user" turn
        # (the Gemini API has no separate "tool" role — function results are
        # sent back on "user", same as Step 8 earlier), then loop back around
        # and ask Gemini again — maybe it's done, maybe it needs another tool.
        conversation.append(types.Content(role="user", parts=turn_response_parts))

    # We tried `max_turns` times and Gemini still wasn't done — bail out safely
    # instead of looping forever (this is the payoff of the cap set above).
    return "⚠️ Reached max_turns without a final answer."

### Why `max_turns` exists

This loop only stops on its own when Gemini answers with plain text. Without a cap, anything that makes Gemini keep requesting tools forever — a confusing tool result, a long dependent chain, a misbehaving model — would turn this into an infinite loop that never returns, burning API calls the whole time.

`max_turns` guarantees the function always returns *something*, even in the worst case:

| Turn | What happens | Loop continues? |
|---|---|---|
| 1 | Ask Gemini → it requests some tools → we run them | Yes, results sent back |
| 2 | Ask Gemini again → it requests *more* tools → we run those | Yes, results sent back |
| 3 | Ask Gemini again → it requests *even more* tools → we run those | Yes, results sent back |
| 4 | Ask Gemini again → it *still* requests tools | **No — loop ends after this** |

If turn `max_turns` still isn't a plain-text answer, the function falls through to a safe fallback message instead of hanging:

```python
return "⚠️ Reached max_turns without a final answer."

## 12. Try it on a few different requests

Including one that needs **zero** tools — proving the loop doesn't force tool use when nothing is required.

In [16]:
import time  # gives us time.sleep(), which just pauses the program for N seconds

# Three different questions to try — testing that the function handles
# different situations correctly.
demo_prompts = [
    "What time is it in Tokyo, and what is 15% of 8500?",
    "Convert 250 USD to EUR, then tell me the weather in Paris, then the time in New York.",
    "Explain what RAG is in one sentence.",
]

prompt_index = 0  # tracks which prompt we're on: 0, 1, 2...

for prompt in demo_prompts:

    # ---------- Pace ourselves so we don't get rate-limited ----------
    # Skip the pause before the very FIRST prompt (prompt_index == 0) —
    # there's nothing to wait for yet. Before every prompt AFTER that,
    # wait 15 seconds first.
    if prompt_index > 0:
        # WHY: Gemini's free tier only allows a limited number of requests
        # per minute. Firing 3 prompts back-to-back (each prompt can itself
        # be 1-4 API calls, remember max_turns) can trip that limit and
        # cause a "429 RESOURCE_EXHAUSTED" error mid-demo. Sleeping between
        # prompts keeps us safely under that limit.
        time.sleep(15)

    # ---------- Run this one prompt through the whole pipeline ----------
    print("=" * 70)        # prints a line of 70 "=" characters, just a visual divider
    print("USER:", prompt)
    print("-" * 70)        # a lighter divider

    answer = run_multi_tool_query(prompt)  # this is where all the real work happens

    print("\nFINAL ANSWER:", answer)
    print()  # blank line, so the next prompt's output doesn't run into this one

    prompt_index += 1  # move on to the next prompt

USER: What time is it in Tokyo, and what is 15% of 8500?
----------------------------------------------------------------------
Saved to: llm_logs\response_20260923_124310_566379.json
--- Turn 1: Gemini requested 2 tool call(s) ---
Step 1: get_current_time({'timezone': 'Asia/Tokyo'}) -> {'timezone': 'Asia/Tokyo', 'time': '2026-09-23 21:43:10'}
Step 2: calculate({'operation': 'percentage', 'b': 8500, 'a': 15}) -> 1275.0
Saved to: llm_logs\response_20260923_124311_787500.json

FINAL ANSWER: The current time in Tokyo is 9:43 PM on September 23, 2026, and 15% of 8500 is 1275.

USER: Convert 250 USD to EUR, then tell me the weather in Paris, then the time in New York.
----------------------------------------------------------------------
Saved to: llm_logs\response_20260923_124342_236288.json
--- Turn 1: Gemini requested 3 tool call(s) ---
Step 1: convert_currency({'to_currency': 'EUR', 'from_currency': 'USD', 'amount': 250}) -> {'amount': 250, 'from': 'USD', 'to': 'EUR', 'converted': 230.0

## 13. What you just built, in one sentence

> **Gemini decides *what* to call and *in what order*; a plain Python dictionary and a `for` loop decide *how* it actually runs.**

That boundary — model plans, application executes — is the entire idea behind every "AI agent" framework. The parts that feel like magic from the outside are, from the inside:

| What it looks like from outside | What it actually is |
|---|---|
| `@tool` decorator | Building a `FunctionDeclaration`, same as Step 2 |
| Agent "decides" to call 3 tools | Gemini returning 3 `function_call` parts, same as Step 6 |
| Framework "runs" the tools | A `{name: callable}` dict lookup, same as Step 5 & 7 |
| Agent "loops until done" | The `for turn in range(max_turns)` loop, same as Step 11 |

### One sentence to remember

> **A "framework" for tool calling is a dictionary and a loop wearing a nicer API.**

### Where this leads next

Tomorrow (Day 4 — Agents & MCP) keeps this exact loop, but instead of a single question with a knowable plan, the model is given an open-ended **goal** and decides the whole sequence of steps itself, one at a time, based on what it observes after each tool result.

## Official references

- Gemini API — Function calling: https://ai.google.dev/gemini-api/docs/function-calling
- Gemini API — Tools: https://ai.google.dev/gemini-api/docs/tools
- Gemini API — Getting started: https://ai.google.dev/gemini-api/docs/get-started
- Google AI Studio: https://aistudio.google.com/